# Riffusion — Spectral Diffusion for Music

Riffusion generates music by treating spectrograms as images and using Stable Diffusion
to generate them. The generated spectrograms are then converted back to audio via
Griffin-Lim or a neural vocoder.

This shows a fundamentally different approach from MusicGen:
- **MusicGen**: token-space generation (text -> audio tokens -> waveform)
- **Riffusion**: image-space generation (text -> spectrogram image -> waveform)

In [ ]:
!pip install diffusers torch matplotlib scipy IPython numpy Pillow torchaudio librosa

In [ ]:
from diffusers import StableDiffusionPipeline
import torch
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd
from PIL import Image
import librosa

pipe = StableDiffusionPipeline.from_pretrained(
    "riffusion/riffusion-model-v1",
    torch_dtype=torch.float16
)
pipe = pipe.to("cuda")
print("Riffusion model loaded.")

## Generate Spectrograms from Text

Riffusion generates 512x512 spectrogram images from text prompts.
These images represent the frequency content of audio over time.

In [ ]:
prompts = [
    "funky bass guitar groove",
    "ethereal ambient pad with reverb",
    "fast rock drum beat with cymbals",
]

spectrograms = []

fig, axes = plt.subplots(1, len(prompts), figsize=(5 * len(prompts), 5))

for i, prompt in enumerate(prompts):
    result = pipe(
        prompt,
        num_inference_steps=50,
        width=512,
        height=512,
    )
    img = result.images[0]
    spectrograms.append(img)
    axes[i].imshow(img)
    axes[i].set_title(prompt, fontsize=10)
    axes[i].axis("off")

plt.suptitle("Riffusion-Generated Spectrograms", fontsize=14)
plt.tight_layout()
plt.show()

## Convert Spectrogram to Audio

We convert the generated spectrogram images back to audio using Griffin-Lim
phase reconstruction. The spectrogram pixel values are interpreted as
magnitude spectrogram bins.

In [ ]:
def spectrogram_image_to_audio(image, sr=44100, n_fft=2048, hop_length=512,
                                n_iter=100, n_mels=256):
    """Convert a Riffusion spectrogram image to audio via Griffin-Lim."""
    # Convert image to grayscale numpy array
    img_array = np.array(image.convert("L")).astype(np.float32)

    # The image is a mel spectrogram: rows = frequency (top=high), cols = time
    # Flip vertically so low frequencies are at the bottom
    img_array = img_array[::-1, :]

    # Normalize to [0, 1] range and convert to dB-like scale
    img_array = img_array / 255.0

    # Resize to n_mels x time_frames
    from PIL import Image as PILImage
    img_resized = PILImage.fromarray((img_array * 255).astype(np.uint8))
    time_frames = img_array.shape[1]
    img_resized = img_resized.resize((time_frames, n_mels))
    mel_spec = np.array(img_resized).astype(np.float32) / 255.0

    # Scale to power spectrogram range
    mel_spec = mel_spec * 80.0 - 80.0  # dB range
    mel_spec = librosa.db_to_power(mel_spec)

    # Create mel filter bank and invert
    mel_basis = librosa.filters.mel(sr=sr, n_fft=n_fft, n_mels=n_mels)
    # Pseudo-inverse to go from mel back to linear
    mel_basis_pinv = np.linalg.pinv(mel_basis)
    linear_spec = np.maximum(0, mel_basis_pinv @ mel_spec)

    # Griffin-Lim phase reconstruction
    audio = librosa.griffinlim(
        np.sqrt(linear_spec),  # amplitude spectrogram
        n_iter=n_iter,
        hop_length=hop_length,
        n_fft=n_fft
    )

    # Normalize
    audio = audio / (np.max(np.abs(audio)) + 1e-8)
    return audio.astype(np.float32)


# Convert each generated spectrogram to audio
sr = 44100

for i, (prompt, spec_img) in enumerate(zip(prompts, spectrograms)):
    audio = spectrogram_image_to_audio(spec_img, sr=sr)
    print(f"\n{prompt} ({len(audio)/sr:.1f}s):")
    ipd.display(ipd.Audio(audio, rate=sr))

## Interpolation Between Styles

One advantage of diffusion models is smooth interpolation in latent space.
We can blend between two text prompts to create smooth style transitions.

In [ ]:
from diffusers import StableDiffusionPipeline

prompt_a = "calm acoustic guitar"
prompt_b = "heavy electric guitar distortion"

# Encode both prompts
tokenizer = pipe.tokenizer
text_encoder = pipe.text_encoder

def get_text_embedding(prompt):
    tokens = tokenizer(prompt, return_tensors="pt", padding="max_length",
                       max_length=tokenizer.model_max_length, truncation=True)
    tokens = {k: v.to(pipe.device) for k, v in tokens.items()}
    with torch.no_grad():
        embedding = text_encoder(tokens["input_ids"])[0]
    return embedding

emb_a = get_text_embedding(prompt_a)
emb_b = get_text_embedding(prompt_b)

# Generate interpolated spectrograms
alphas = [0.0, 0.25, 0.5, 0.75, 1.0]
interp_images = []

fig, axes = plt.subplots(1, len(alphas), figsize=(4 * len(alphas), 4))

for i, alpha in enumerate(alphas):
    # Spherical linear interpolation (slerp) for embeddings
    interp_emb = (1 - alpha) * emb_a + alpha * emb_b
    interp_emb = interp_emb / interp_emb.norm(dim=-1, keepdim=True) * emb_a.norm(dim=-1, keepdim=True)

    # Generate with interpolated embedding
    generator = torch.Generator(device=pipe.device).manual_seed(42)
    result = pipe(
        prompt_embeds=interp_emb,
        num_inference_steps=50,
        width=512, height=512,
        generator=generator,
    )
    img = result.images[0]
    interp_images.append(img)

    axes[i].imshow(img)
    if alpha == 0.0:
        axes[i].set_title(f"'{prompt_a}'\nalpha={alpha}", fontsize=9)
    elif alpha == 1.0:
        axes[i].set_title(f"'{prompt_b}'\nalpha={alpha}", fontsize=9)
    else:
        axes[i].set_title(f"alpha={alpha}", fontsize=9)
    axes[i].axis("off")

plt.suptitle("Interpolation: Acoustic Guitar -> Distorted Guitar", fontsize=13)
plt.tight_layout()
plt.show()

# Convert interpolations to audio
for i, (alpha, img) in enumerate(zip(alphas, interp_images)):
    audio = spectrogram_image_to_audio(img, sr=sr)
    print(f"\nalpha={alpha}:")
    ipd.display(ipd.Audio(audio, rate=sr))

## Comparison: Riffusion vs MusicGen

Let's compare the same prompts through both systems to hear the differences
in generation approach.

| Aspect | MusicGen | Riffusion |
|--------|----------|----------|
| **Domain** | Audio tokens (EnCodec) | Spectrogram images |
| **Architecture** | Transformer language model | Stable Diffusion (U-Net) |
| **Phase** | Preserved (EnCodec) | Reconstructed (Griffin-Lim) |
| **Duration** | Variable (token count) | Fixed (image size) |
| **Quality** | Generally higher fidelity | Can sound "washy" |
| **Interpolation** | Not straightforward | Smooth latent interpolation |

In [ ]:
# Compare both approaches on the same prompts
# Load MusicGen for comparison
from transformers import AutoProcessor, MusicgenForConditionalGeneration

mg_processor = AutoProcessor.from_pretrained("facebook/musicgen-small")
mg_model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-small")
mg_model = mg_model.to("cuda" if torch.cuda.is_available() else "cpu")
mg_sr = mg_model.config.audio_encoder.sampling_rate

comparison_prompts = [
    "Upbeat funk with slap bass",
    "Soft piano ballad with strings",
]

for prompt in comparison_prompts:
    print(f"\n{'='*60}")
    print(f"Prompt: '{prompt}'")
    print(f"{'='*60}")

    # MusicGen
    mg_inputs = mg_processor(text=[prompt], padding=True, return_tensors="pt").to(mg_model.device)
    with torch.no_grad():
        mg_audio = mg_model.generate(**mg_inputs, max_new_tokens=512)
    print("\nMusicGen:")
    ipd.display(ipd.Audio(mg_audio[0].cpu().numpy(), rate=mg_sr))

    # Riffusion
    result = pipe(prompt, num_inference_steps=50, width=512, height=512)
    riff_audio = spectrogram_image_to_audio(result.images[0], sr=sr)
    print("\nRiffusion:")
    ipd.display(ipd.Audio(riff_audio, rate=sr))